In [ ]:

import pandas as pd
import nltk
import re

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

nltk.download('stopwords')

df = pd.read_csv(r"Reviews.csv", nrows=50000)

print("Dataset Shape:", df.shape)

print(df.head())
df = df[['Text', 'Score']]

df.dropna(inplace=True)

print("\nDataset after selecting required columns:")
print(df.head())

def get_sentiment(score):
    if score <= 2:
        return "Negative"
    elif score == 3:
        return "Neutral"
    else:
        return "Positive"

df["Sentiment"] = df["Score"].apply(get_sentiment)

df = df[['Text', 'Sentiment']]
df.columns = ['Review', 'Sentiment']

print("\nSentiment Distribution")
print(df['Sentiment'].value_counts())

stop_words = set(stopwords.words('english'))

stemmer = PorterStemmer()

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = text.split()

    words = [stemmer.stem(word)
             for word in words
             if word not in stop_words]

    # Join words
    return " ".join(words)

print("\nCleaning text...")

df["Clean_Text"] = df["Review"].apply(preprocess)

print(df[['Review','Clean_Text']].head())

tfidf = TfidfVectorizer(max_features=5000)

X = tfidf.fit_transform(df["Clean_Text"])

y = df["Sentiment"]

print("\nFeature Matrix Shape:", X.shape)


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("\nTraining Samples :", X_train.shape[0])
print("Testing Samples  :", X_test.shape[0])


model = MultinomialNB()

model.fit(X_train, y_train)

print("\nModel Trained Successfully!")


prediction = model.predict(X_test)

print("\nFirst 20 Predictions")
print(prediction[:20])


accuracy = accuracy_score(y_test, prediction)

print("\nAccuracy Score")
print(accuracy)


print("\nClassification Report")
print(classification_report(y_test, prediction))


print("\nConfusion Matrix")
print(confusion_matrix(y_test, prediction))


reviews = [

    "This product is amazing and worth every penny.",

    "Worst product I have ever purchased.",

    "The quality is average.",

    "Excellent customer service and fast delivery.",

    "Very disappointing experience.",

    "I absolutely love this item.",

    "Waste of money.",

    "It works fine and meets expectations.",

    "Fantastic taste and premium quality.",

    "Terrible packaging and damaged product."

]

clean_reviews = [preprocess(review) for review in reviews]

review_vectors = tfidf.transform(clean_reviews)

results = model.predict(review_vectors)

print("\n================= Prediction Results =================")

for review, sentiment in zip(reviews, results):
    print("Review    :", review)
    print("Sentiment :", sentiment)
    print("-"*70)

[nltk_data] Downloading package stopwords to C:\Users\Raj
[nltk_data]     Coach\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Dataset Shape: (50000, 10)
   Id   ProductId          UserId                      ProfileName  \
0   1  B001E4KFG0  A3SGXH7AUHU8GW                       delmartian   
1   2  B00813GRG4  A1D87F6ZCVE5NK                           dll pa   
2   3  B000LQOCH0   ABXLMWJIXXAIN  Natalia Corres "Natalia Corres"   
3   4  B000UA0QIQ  A395BORC6FGVXV                             Karl   
4   5  B006K2ZZ7K  A1UQRSCLF8GW1T    Michael D. Bigham "M. Wassir"   

   HelpfulnessNumerator  HelpfulnessDenominator  Score        Time  \
0                     1                       1      5  1303862400   
1                     0                       0      1  1346976000   
2                     1                       1      4  1219017600   
3                     3                       3      2  1307923200   
4                     0                       0      5  1350777600   

                 Summary                                               Text  
0  Good Quality Dog Food  I have bought several of th